# Waste Classification - Data Preparation & EDA

Full data pipeline:
1. **Download** 3 public datasets from Kaggle
2. **Explore** each dataset structure (classes, image counts)
3. **Unify** classes across datasets into a common mapping
4. **EDA** — class distribution, sample images, image sizes, pixel statistics
5. **Balance** classes by capping overrepresented ones
6. **Split** into train/val/test (80/10/10)
7. **Save** preprocessed data for training

This notebook prepares the data used by `model-r.ipynb` for training the ResNet50 classifier.

### Imports

- **`os`**: interact with the file system (list directories, create folders, check paths).
- **`shutil`**: high-level file operations (copy files between directories).
- **`random`**: pseudo-random number generation. Used for shuffling and reproducible splits.
- **`numpy`** (`np`): numerical array operations. Used for pixel analysis and statistics.
- **`matplotlib.pyplot`** (`plt`): visualization library. Creates charts (histograms, bars, grids).
- **`seaborn`** (`sns`): statistical visualization built on matplotlib. Used for heatmaps.
- **`PIL.Image`**: library to open, resize, and manipulate images.
- **`kagglehub`**: Kaggle API wrapper to download datasets directly into the notebook environment.

In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import kagglehub

## 1. Download Datasets

We download 3 public waste classification datasets from Kaggle using `kagglehub`.

- **`kagglehub.dataset_download(handle)`**: downloads the dataset and returns the local path where files are stored.
- Each dataset is organized in folders where each folder name = class name and contains images of that class.
- **Why 3 datasets?** Combining multiple sources increases diversity: different backgrounds, lighting conditions, camera angles, and object variations. This helps the model generalize better to real-world images.

| Dataset | Handle | Description |
|---------|--------|-------------|
| Garbage Classification | `asdasdasasdas/garbage-classification` | 6 classes, ~2,500 images |
| Garbage Classification (12 classes) | `mostafaabla/garbage-classification` | 12 classes, ~15,500 images |
| RealWaste | `joebeachcapital/realwaste` | 9 classes, ~4,750 images |

In [ ]:
path1 = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
print("Dataset 1:", path1)

path2 = kagglehub.dataset_download("mostafaabla/garbage-classification")
print("Dataset 2:", path2)

path3 = kagglehub.dataset_download("joebeachcapital/realwaste")
print("Dataset 3:", path3)

## 2. Dataset Exploration

Before any processing, we explore the raw structure of each dataset to understand:
- How many classes exist in each dataset
- How many images per class
- Whether class names are consistent across datasets

- **`os.walk(path)`**: recursively traverses a directory tree. Returns `(root, dirs, files)` for each directory.
- **`if files and not dirs`**: filters leaf directories (folders that contain files but no subdirectories). These are the class folders.
- **`os.path.basename(root)`**: extracts just the folder name from the full path (this is the class name).

In [ ]:
datasets = {
    "garbage_classification": path1,
    "garbage_12_classes": path2,
    "realwaste": path3,
}

for name, path in datasets.items():
    print(f"\n{'='*40}")
    print(f"Dataset: {name}")
    total = 0
    for root, dirs, files in os.walk(path):
        if files and not dirs:
            class_name = os.path.basename(root)
            count = len(files)
            total += count
            print(f"  {class_name:20s} -> {count:>5}")
    print(f"  {'TOTAL':20s} -> {total:>5}")

### Key Observations

The 3 datasets use different class names and granularities:

- **garbage_classification**: 6 broad classes (cardboard, glass, metal, paper, plastic, trash)
- **garbage_12_classes**: 12 finer classes (splits glass into brown/green/white-glass, adds battery, biological, clothes, shoes)
- **realwaste**: 9 classes with different naming (Cardboard vs cardboard, Food Organics, Vegetation, Textile Trash)

We need a **unified class mapping** to merge them.

## 3. Dataset Unification

We define a mapping that converts each dataset's class names into a unified set of 7 classes.

### Mapping Rules

| Original Class | Source Dataset | Unified Class |
|---------------|---------------|---------------|
| cardboard | garbage_classification | cardboard |
| glass | garbage_classification | glass |
| metal | garbage_classification | metal |
| paper | garbage_classification | paper |
| plastic | garbage_classification | plastic |
| trash | garbage_classification | trash |
| brown-glass, green-glass, white-glass | garbage_12_classes | glass |
| biological | garbage_12_classes | organic |
| Cardboard, Glass, Metal, Paper, Plastic | realwaste | (lowercase equivalent) |
| Miscellaneous Trash | realwaste | trash |
| Food Organics, Vegetation | realwaste | organic |

### Excluded Classes

- **battery** (only in garbage_12_classes) — no equivalent in other datasets
- **shoes** (only in garbage_12_classes) — no equivalent in other datasets
- **clothes / Textile Trash** — initially included as `textile` but later removed because it was heavily overrepresented (5,643 images vs ~2,000 average) and only present in 2 datasets

### Implementation

- **`CLASS_MAP`**: dictionary mapping each original class name to its unified name. Classes not in the map are skipped.
- **`shutil.copy2(src, dst)`**: copies a file preserving metadata (timestamps). We use this instead of `shutil.copy()` to maintain file attributes.
- **Filename prefixing**: `f"{dataset_name}_{original_class}_{f}"` prevents name collisions when images from different datasets have the same filename (e.g., `image001.jpg`).

In [ ]:
CLASS_MAP = {
    # garbage_classification (6 classes)
    "cardboard": "cardboard", "glass": "glass", "metal": "metal",
    "paper": "paper", "plastic": "plastic", "trash": "trash",
    # garbage_12_classes — glass variants merged
    "brown-glass": "glass", "green-glass": "glass", "white-glass": "glass",
    # garbage_12_classes — biological -> organic
    "biological": "organic",
    # realwaste — capitalized names
    "Cardboard": "cardboard", "Glass": "glass", "Metal": "metal",
    "Paper": "paper", "Plastic": "plastic",
    "Miscellaneous Trash": "trash",
    "Food Organics": "organic",
    "Vegetation": "organic",
}

UNIFIED_PATH = "/kaggle/working/unified_dataset"
counter = {}

for dataset_name, dataset_path in datasets.items():
    for root, dirs, files in os.walk(dataset_path):
        if files and not dirs:
            original_class = os.path.basename(root)
            if original_class not in CLASS_MAP:
                continue  # Skip battery, shoes, clothes, Textile Trash
            unified_class = CLASS_MAP[original_class]
            dest_dir = os.path.join(UNIFIED_PATH, unified_class)
            os.makedirs(dest_dir, exist_ok=True)
            for f in files:
                src = os.path.join(root, f)
                new_name = f"{dataset_name}_{original_class}_{f}"
                dst = os.path.join(dest_dir, new_name)
                shutil.copy2(src, dst)
                counter[unified_class] = counter.get(unified_class, 0) + 1

print("Unified dataset:")
for cls, count in sorted(counter.items()):
    print(f"  {cls:15s} -> {count:>5}")
print(f"  {'TOTAL':15s} -> {sum(counter.values()):>5}")

## 4. Exploratory Data Analysis (EDA)

### 4.1 Class Distribution (Before Balancing)

We visualize how many images each unified class has before any balancing.

- **`os.listdir(path)`**: returns a list of all files and folders in `path`.
- **`plt.bar(classes, counts)`**: creates a bar chart where each bar represents a class.
- **`plt.bar_label(bars)`**: displays the exact count on top of each bar.
- **Imbalance check**: if some classes have significantly more images than others, the model may become biased towards overrepresented classes.

In [ ]:
classes = sorted(os.listdir(UNIFIED_PATH))
counts = [len(os.listdir(os.path.join(UNIFIED_PATH, c))) for c in classes]

plt.figure(figsize=(10, 5))
bars = plt.bar(classes, counts, color='#52b788', edgecolor='white')
plt.bar_label(bars, fontweight='bold')
plt.title('Unified Dataset - Class Distribution (Before Balancing)', fontsize=13, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nTotal images: {sum(counts)}")
print(f"Min class: {classes[np.argmin(counts)]} ({min(counts)})")
print(f"Max class: {classes[np.argmax(counts)]} ({max(counts)})")
print(f"Imbalance ratio: {max(counts)/min(counts):.1f}x")

### 4.2 Sample Images per Class

We display one sample image per class to visually verify the data is correct and understand what the model will see.

- **`plt.subplots(2, 4)`**: creates a 2x4 grid of subplots (7 classes + 1 empty).
- **`axes.flat`**: converts the 2D axes matrix into a flat list for easy iteration.
- **`Image.open(path)`**: opens an image file as a PIL Image object.
- **`ax.imshow(img)`**: displays the image inside a matplotlib subplot.
- **`ax.axis('off')`**: hides the axis ticks and labels (cleaner visualization).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, cls in zip(axes.flat, classes):
    cls_path = os.path.join(UNIFIED_PATH, cls)
    img_name = os.listdir(cls_path)[0]
    img = Image.open(os.path.join(cls_path, img_name))
    ax.imshow(img)
    ax.set_title(cls, fontsize=12, fontweight='bold')
    ax.axis('off')

# Hide the 8th subplot (only 7 classes)
if len(classes) < len(axes.flat):
    axes.flat[-1].axis('off')

plt.suptitle('Sample Image per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Image Size Distribution

We analyze the original sizes of images across the dataset. This is important because:
- The model expects a **fixed input size** (224x224 pixels).
- Images with very different aspect ratios may lose information when resized.
- Understanding the size distribution helps decide the target resolution.

- **`img.size`**: returns `(width, height)` of a PIL Image.
- **Sampling 100 per class**: analyzing every image would be slow; 100 per class gives a representative distribution.
- **`plt.hist()`**: creates a histogram showing the frequency of different values.

In [ ]:
widths, heights = [], []
for cls in classes:
    cls_path = os.path.join(UNIFIED_PATH, cls)
    for f in os.listdir(cls_path)[:100]:  # Sample 100 per class
        img = Image.open(os.path.join(cls_path, f))
        w, h = img.size
        widths.append(w)
        heights.append(h)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=30, color='#2d6a4f', edgecolor='white')
ax1.set_title('Image Width Distribution', fontweight='bold')
ax1.set_xlabel('Width (px)')
ax1.set_ylabel('Count')

ax2.hist(heights, bins=30, color='#52b788', edgecolor='white')
ax2.set_title('Image Height Distribution', fontweight='bold')
ax2.set_xlabel('Height (px)')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"\nTarget resize: 224x224 (standard for ResNet50)")

### 4.4 Pixel Statistics

We analyze the pixel value distribution per color channel (RGB). This helps understand:
- Whether the data needs special normalization
- If there are brightness or color biases across the dataset
- The expected input range for the model

- **`np.random.seed(42)`**: fixes the random seed for reproducibility.
- **`img.resize((224, 224))`**: resizes to the target model input size before analysis.
- **`np.array(img)`**: converts the PIL Image to a NumPy array of shape `(224, 224, 3)` with values in `[0, 255]`.
- **Channel indexing** `[:, :, ch]`: selects all pixels of a single color channel (0=Red, 1=Green, 2=Blue).

In [ ]:
np.random.seed(42)

# Collect 500 random images resized to 224x224
all_files = []
for cls in classes:
    cls_path = os.path.join(UNIFIED_PATH, cls)
    for f in os.listdir(cls_path):
        all_files.append(os.path.join(cls_path, f))

sample_paths = np.random.choice(all_files, size=500, replace=False)
sample_pixels = []
for path in sample_paths:
    img = Image.open(path).convert('RGB').resize((224, 224))
    sample_pixels.append(np.array(img))
sample_pixels = np.array(sample_pixels).astype(np.float32)

print(f"Sample shape: {sample_pixels.shape}")
print(f"dtype: {sample_pixels.dtype}")
print(f"Pixel range: [{sample_pixels.min():.0f}, {sample_pixels.max():.0f}]")

# Per-channel statistics
channel_names = ['Red', 'Green', 'Blue']
channel_colors = ['#e63946', '#2d6a4f', '#1d3557']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ch, (ax, name, color) in enumerate(zip(axes, channel_names, channel_colors)):
    values = sample_pixels[:, :, :, ch].flatten()
    ax.hist(values, bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'{name} Channel', fontweight='bold')
    ax.set_xlabel('Pixel Value')
    ax.set_ylabel('Frequency')
    ax.axvline(values.mean(), color='black', linestyle='--', linewidth=1, label=f'mean={values.mean():.1f}')
    ax.legend(fontsize=9)

plt.suptitle('Pixel Value Distribution per Channel (500 samples)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
for ch, name in enumerate(channel_names):
    values = sample_pixels[:, :, :, ch]
    print(f"{name:5s} — mean: {values.mean():.1f}, std: {values.std():.1f}, min: {values.min():.0f}, max: {values.max():.0f}")

### 4.5 Dataset Source Contribution per Class

We visualize how much each source dataset contributes to each unified class. This shows the diversity of data sources per class.

- Images from `garbage_classification` are prefixed with `garbage_classification_`
- Images from `garbage_12_classes` are prefixed with `garbage_12_classes_`
- Images from `realwaste` are prefixed with `realwaste_`

In [ ]:
source_counts = {cls: {'garbage_classification': 0, 'garbage_12_classes': 0, 'realwaste': 0} for cls in classes}

for cls in classes:
    cls_path = os.path.join(UNIFIED_PATH, cls)
    for f in os.listdir(cls_path):
        for src in ['garbage_classification', 'garbage_12_classes', 'realwaste']:
            if f.startswith(src):
                source_counts[cls][src] += 1
                break

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(classes))
width = 0.25
colors = ['#2d6a4f', '#52b788', '#b7e4c7']
sources = ['garbage_classification', 'garbage_12_classes', 'realwaste']
labels = ['Garbage Class. (6)', 'Garbage Class. (12)', 'RealWaste']

for i, (src, label, color) in enumerate(zip(sources, labels, colors)):
    vals = [source_counts[cls][src] for cls in classes]
    bars = ax.bar(x + i * width, vals, width, label=label, color=color, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(classes, rotation=45)
ax.set_title('Dataset Source Contribution per Class', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Images')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Train / Val / Test Split

We split the unified dataset into 3 sets:

- **Train (80%)**: the model learns from these images.
- **Validation (10%)**: used during training to detect overfitting. The model never trains on these, but we check accuracy after each epoch.
- **Test (10%)**: used only once at the very end for final evaluation. The model never sees these during training.

### Balancing

- **`MAX_PER_CLASS = 2500`**: caps overrepresented classes. Without this, `glass` (3,433 images) would dominate training.
- **`random.seed(42)`**: ensures the split is reproducible — running this cell again produces the same split.
- **`random.shuffle(files)`**: randomizes file order before splitting to avoid any ordering bias.

### Split Calculation

- **`n_train = int(n * 0.8)`**: 80% of files go to train.
- **`n_val = int(n * 0.1)`**: 10% go to validation.
- **Remaining files go to test**: this ensures no images are lost due to rounding.

In [ ]:
SPLIT_PATH = "/kaggle/working/dataset_split"
MAX_PER_CLASS = 2500
SEED = 42
random.seed(SEED)

train_count, val_count, test_count = {}, {}, {}

for cls in sorted(os.listdir(UNIFIED_PATH)):
    cls_path = os.path.join(UNIFIED_PATH, cls)
    files = [f for f in os.listdir(cls_path) if not f.startswith('.')]
    random.shuffle(files)
    files = files[:MAX_PER_CLASS]  # Cap overrepresented classes
    
    n = len(files)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    
    splits = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:],
    }
    
    for split_name, split_files in splits.items():
        dest = os.path.join(SPLIT_PATH, split_name, cls)
        os.makedirs(dest, exist_ok=True)
        for f in split_files:
            shutil.copy2(os.path.join(cls_path, f), os.path.join(dest, f))
    
    train_count[cls] = len(splits["train"])
    val_count[cls] = len(splits["val"])
    test_count[cls] = len(splits["test"])

print(f"{'Class':15s} {'Train':>6} {'Val':>6} {'Test':>6}")
print("-" * 35)
for cls in sorted(train_count):
    print(f"{cls:15s} {train_count[cls]:>6} {val_count[cls]:>6} {test_count[cls]:>6}")
print("-" * 35)
print(f"{'TOTAL':15s} {sum(train_count.values()):>6} {sum(val_count.values()):>6} {sum(test_count.values()):>6}")

### 5.1 Split Distribution Visualization

We visualize the class distribution across all 3 splits to verify:
- Each split maintains similar class proportions (stratified split)
- No class is under or overrepresented in any split

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
split_data = [
    ('Train', train_count, '#2d6a4f'),
    ('Validation', val_count, '#52b788'),
    ('Test', test_count, '#b7e4c7')
]

for ax, (name, data, color) in zip(axes, split_data):
    sorted_classes = sorted(data.keys())
    vals = [data[c] for c in sorted_classes]
    bars = ax.bar(sorted_classes, vals, color=color, edgecolor='white')
    ax.bar_label(bars, fontweight='bold', fontsize=9)
    ax.set_title(f'{name} Set ({sum(vals)} images)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Images')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Data Quality Checks

Before saving, we verify data integrity:
- All images can be opened without errors (no corrupted files)
- All images have 3 channels (RGB) — no grayscale or RGBA images
- No empty class folders

In [ ]:
corrupted = 0
non_rgb = 0
total_checked = 0

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(SPLIT_PATH, split)
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        files = os.listdir(cls_path)
        for f in files[:50]:  # Check 50 per class per split
            total_checked += 1
            try:
                img = Image.open(os.path.join(cls_path, f))
                if img.mode != 'RGB':
                    non_rgb += 1
            except Exception:
                corrupted += 1

print(f"Images checked: {total_checked}")
print(f"Corrupted:      {corrupted}")
print(f"Non-RGB:        {non_rgb}")
print(f"\nData quality: {'PASS' if corrupted == 0 else 'ISSUES FOUND'}")

## 7. Summary

### Data Preparation Pipeline

```
3 Kaggle datasets (22,821 total images)
    |
    v
Unified into 7 classes (16,756 images)
    |
    v
Balanced (max 2,500 per class)
    |
    v
Split: Train 11,036 | Val 1,376 | Test 1,385
    |
    v
Ready for model training (model-r.ipynb)
```

### Final Classes

| Index | Class | Description |
|-------|-------|-------------|
| 0 | cardboard | Cardboard boxes, packaging |
| 1 | glass | Bottles, jars (brown, green, white glass merged) |
| 2 | metal | Cans, aluminum, metal containers |
| 3 | organic | Food waste, vegetation, biological matter |
| 4 | paper | Paper sheets, newspapers, documents |
| 5 | plastic | Plastic bottles, bags, containers |
| 6 | trash | Mixed waste, non-recyclable items |

### Key Decisions

- **Textile excluded**: heavily overrepresented (5,643 images) and only in 2 of 3 datasets.
- **Glass merged**: brown, green, and white glass combined into one class — the recycling process is the same.
- **Max 2,500 per class**: prevents model bias towards overrepresented classes.
- **224x224 target size**: standard input size for ResNet50 (chosen model).
- **No pixel normalization here**: preprocessing is model-specific and applied in the training pipeline.